# Baseline Colab — MOT15 + MOT16

Этап 1: генерация `.npy` (det + ReID) и прогон оригинального DeepSORT на **6 sequences**.  
Оба набора генерируются одинаково через `generate_detections.py` → `MOT15_train/` и `MOT16_train/`.

In [ ]:
DRIVE_RESOURCES = "/content/drive/MyDrive/DeepSORT/resources"

from google.colab import drive
drive.mount("/content/drive")

import os
assert os.path.isdir(DRIVE_RESOURCES), f"Папка не найдена: {DRIVE_RESOURCES}"

# Клонируем код (HTTPS, без SSH)
if not os.path.isdir("/content/DeepSORT_Project_CV"):
    !git clone https://github.com/Valeriia-Reznik-Dev/DeepSORT_Project_CV.git /content/DeepSORT_Project_CV

%cd /content/DeepSORT_Project_CV
!git pull

# Подключаем ваши данные с Drive
if os.path.islink("resources"):
    os.remove("resources")
elif os.path.isdir("resources"):
    !rm -rf resources
!ln -s "{DRIVE_RESOURCES}" resources

print("resources ->", os.path.realpath("resources"))

In [ ]:
!pip install -q "numpy>=2.0" opencv-python scipy pyyaml

import numpy as np
print("NumPy", np.__version__)
assert np.__version__.split(".")[0] >= "2", (
    "numpy<2 обнаружен. Перезапустите рантайм (Runtime -> Restart session) и выполните ячейку заново."
)

import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPU", tf.config.list_physical_devices("GPU"))


In [ ]:
import os

MOT15_SEQUENCES = ["TUD-Campus", "TUD-Stadtmitte", "KITTI-17", "PETS09-S2L1"]
MOT16_SEQUENCES = ["MOT16-09", "MOT16-11"]
DRIVE_RESOURCES = "/content/drive/MyDrive/DeepSORT/resources"

paths = {
    "model": os.path.join(DRIVE_RESOURCES, "networks", "mars-small128.pb"),
    "mot15_dir": "resources/detections/MOT15/train",
    "mot16_dir": "resources/detections/MOT16/train",
    "mot15_npy_dir": "resources/detections/MOT15_train",
    "mot16_npy_dir": "resources/detections/MOT16_train",
}

assert os.path.isfile(paths["model"]), f"Missing: {paths['model']}"

for seq in MOT15_SEQUENCES:
    assert os.path.isdir(os.path.join(paths["mot15_dir"], seq)), f"Missing MOT15 video: {seq}"
for seq in MOT16_SEQUENCES:
    assert os.path.isdir(os.path.join(paths["mot16_dir"], seq)), f"Missing MOT16 video: {seq}"

print("OK: model + MOT15/MOT16 videos on Drive")

In [ ]:
import os
import time
from google.colab import drive

DRIVE_ROOT = "/content/drive"
DRIVE_RESOURCES = "/content/drive/MyDrive/DeepSORT/resources"
SRC_MODEL = os.path.join(DRIVE_RESOURCES, "networks", "mars-small128.pb")
MODEL = "/content/mars-small128.pb"


def ensure_drive() -> None:
    """Remount Drive if the FUSE mount is dead (OSError 107)."""
    try:
        with open(SRC_MODEL, "rb") as f:
            f.read(1)
    except OSError as exc:
        print(f"Drive mount unhealthy ({exc}); remounting ...")
        drive.mount(DRIVE_ROOT, force_remount=True)


def copy_from_drive(src: str, dst: str, *, chunk_size: int = 1024 * 1024, retries: int = 5) -> None:
    """Copy a Drive file to local disk; remount Drive on FUSE disconnect (errno 107)."""
    ensure_drive()
    src_size = os.path.getsize(src)
    if os.path.isfile(dst) and os.path.getsize(dst) == src_size:
        print(f"Already cached: {dst}")
        return

    last_err = None
    for attempt in range(1, retries + 1):
        try:
            with open(src, "rb") as fsrc, open(dst, "wb") as fdst:
                while True:
                    chunk = fsrc.read(chunk_size)
                    if not chunk:
                        break
                    fdst.write(chunk)
            if os.path.getsize(dst) == src_size:
                print(f"Copied {src_size / 1e6:.1f} MB -> {dst}")
                return
            last_err = OSError(f"size mismatch: got {os.path.getsize(dst)}, expected {src_size}")
        except OSError as exc:
            last_err = exc
            if os.path.isfile(dst):
                os.remove(dst)
            print(f"Drive copy failed ({exc}); remount + retry {attempt}/{retries} in 5s ...")
            time.sleep(5)
            ensure_drive()
            src_size = os.path.getsize(src)

    raise OSError(
        f"Could not copy {src} -> {dst}: {last_err}. "
        "Remount Drive: Runtime -> Restart session, rerun from cell 1, "
        "or drive.mount('/content/drive', force_remount=True)."
    )


copy_from_drive(SRC_MODEL, MODEL)
print("MODEL ->", MODEL)

# Кадры ниже читаются через симлинк resources -> Drive; убедимся, что монтирование живо.
ensure_drive()

# MOT15 + MOT16: один pipeline, generate_detections.py
!python tools/generate_detections.py \
    --model="{MODEL}" \
    --mot_dir=resources/detections/MOT15/train \
    --output_dir=resources/detections/MOT15_train

!python tools/generate_detections.py \
    --model="{MODEL}" \
    --mot_dir=resources/detections/MOT16/train \
    --output_dir=resources/detections/MOT16_train

In [ ]:
import numpy as np

ALL_SEQUENCES = [
    (paths["mot15_npy_dir"], MOT15_SEQUENCES),
    (paths["mot16_npy_dir"], MOT16_SEQUENCES),
]

for npy_dir, seqs in ALL_SEQUENCES:
    print(f"--- {npy_dir}")
    for seq in seqs:
        path = os.path.join(npy_dir, f"{seq}.npy")
        assert os.path.isfile(path), f"Missing: {path}"
        print(seq, np.load(path).shape)

print("OK: all 6 .npy ready")

In [ ]:
!python scripts/run_baseline.py


In [ ]:
# Этап 2: TrackEval HOTA на baseline results
!pip install -q "trackeval==1.3.0"

!python scripts/run_eval.py

## Этап 3 — Detector F1 (YOLO + NanoDet + MMDet)

Сравнение детекций с GT: Precision / Recall / F1 для **всех трёх** детекторов.  
Сначала smoke test (`--max-frames 30`), потом полный прогон без флага.

In [ ]:
!python scripts/setup_detectors_colab.py
# веса NanoDet + RTMDet (если ещё нет в resources/models/)
!python scripts/download_detector_models.py

# Полный прогон: Precision / Recall / F1 + FPS по всем 6 видео.
!python scripts/run_detector_eval.py --detector yolo nanodet mmdet

## Этап 4 — ReID clustering metrics (OSNet + ResNet50-IBN + fast-reid)

Standalone-оценка на **GT pedestrian crops**: Silhouette, Calinski–Harabasz, Fowlkes–Mallows.
Сначала smoke test (`--max-frames 30 --max-samples 500`), потом полный прогон.

In [ ]:
!git pull
!python scripts/setup_reid_colab.py   # also repairs mmcv/torch if MMDet was installed
!python scripts/download_reid_models.py

# если mmdet всё ещё падает с mmcv _ext undefined symbol:
# !python scripts/setup_detectors_colab.py --repair-mmcv-only

# smoke test — все 3 ReID модели
!python scripts/run_reid_eval.py --model osnet resnet50_ibn fastreid --max-frames 30 --max-samples 500

# полный прогон
!python scripts/run_reid_eval.py --model osnet resnet50_ibn fastreid

## Этап 5 — модифицированный DeepSORT (детектор + ReID) → HOTA

Live-пайплайн: детектор → ReID-дескрипторы → ядро DeepSORT (Kalman + matching cascade) → MOT-txt.


In [ ]:
!git pull

NAME = "yolo_osnet"
# Прогон трекера по 6 видео (печатает FPS по каждому).
!python scripts/run_tracker.py --detector yolo --reid osnet --tracker-name {NAME}

# HOTA модифицированного трекера 
!python scripts/run_eval.py --tracker-name {NAME} --results-dir results/tracking/{NAME}

In [ ]:
!zip -r baseline_outputs.zip \
    resources/detections/MOT15_train/ \
    resources/detections/MOT16_train/ \
    results/baseline/original/

from google.colab import files
files.download("baseline_outputs.zip")
